# Tarefa 2 - IMDb Top 250

Versao simplificada da tarefa 2.

A ideia aqui e:
- abrir a lista Top 250 do IMDb;
- pegar os links dos filmes;
- abrir a pagina de cada filme em nova aba;
- extrair os dados pedidos;
- salvar tudo em JSON.


In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import json
import re
import time

# Se quiser testar mais rapido, troque 250 por 5.
TOTAL_FILMES = 250
ARQUIVO_JSON = "filmes_imdb.json"

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

driver.get("https://www.imdb.com/pt/chart/top/")

filmes = wait.until(
    EC.presence_of_all_elements_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

print("Quantidade de cards encontrados:", len(filmes))

# Guardar a aba principal para sempre conseguir voltar para a lista.
aba_principal = driver.current_window_handle

dados = []


Quantidade de cards encontrados: 250


In [10]:
for posicao, filme in enumerate(filmes[:TOTAL_FILMES], start=1):
    try:
        url = filme.find_element(By.TAG_NAME, "a").get_attribute("href")
    except:
        url = None

    if url is None:
        continue

    print(f"Coletando {posicao}/{TOTAL_FILMES}: {url}")

    titulo = None
    ano = None
    nota = None
    url_poster = None
    imagem_poster = None
    lista_generos = []
    lista_diretores = []

    # Abre a pagina do filme em nova aba.
    driver.execute_script("window.open(arguments[0]);", url)
    driver.switch_to.window(driver.window_handles[-1])

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

        # TITULO
        try:
            titulo = driver.find_element(By.TAG_NAME, "h1").text.strip()
        except:
            titulo = None

        # ANO
        try:
            meta_items = driver.find_elements(By.CSS_SELECTOR, 'ul[data-testid="hero-title-block__metadata"] li')
            for item in meta_items:
                texto = item.text.strip()
                if re.fullmatch(r"\d{4}", texto):
                    ano = texto
                    break
        except:
            ano = None

        # NOTA IMDb
        try:
            nota = driver.find_element(
                By.CSS_SELECTOR,
                'div[data-testid="hero-rating-bar__aggregate-rating__score"] span'
            ).text.strip()
        except:
            nota = None

        # POSTER
        try:
            poster = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="hero-media__poster"] img')
            url_poster = poster.get_attribute("src")
            imagem_poster = poster.get_attribute("src")
        except:
            url_poster = None
            imagem_poster = None

        # GENEROS
        try:
            generos = wait.until(
                EC.presence_of_all_elements_located((
                    By.CSS_SELECTOR,
                    ".ipc-chip-list__scroller"
                ))
            )

            for genero in generos:
                tags_genero = genero.find_elements(By.CSS_SELECTOR, "span.ipc-chip__text")

                for tag in tags_genero:
                    texto = tag.text.strip()
                    if texto != "" and texto not in lista_generos:
                        lista_generos.append(texto)
        except:
            lista_generos = []

        # DIRETORES
        try:
            creditos = driver.find_elements(
                By.CSS_SELECTOR,
                'li[data-testid="title-pc-principal-credit"]'
            )

            if len(creditos) > 0:
                # Supondo que o primeiro bloco seja o de direção
                bloco_direcao = creditos[0]
                tags_diretor = bloco_direcao.find_elements(
                    By.CSS_SELECTOR,
                    'a.ipc-metadata-list-item__list-content-item--link'
                )

                for tag in tags_diretor:
                    texto = tag.text.strip()
                    if texto != "" and texto not in lista_diretores:
                        lista_diretores.append(texto)
        except:
            lista_diretores = []

    except TimeoutException:
        print("Tempo esgotado nessa pagina.")

    finally:
        driver.close()
        driver.switch_to.window(aba_principal)

    dados.append({
        "Posicao": posicao,
        "Titulo": titulo,
        "Ano": ano,
        "URL_Poster": url_poster,
        "Imagem_Poster": imagem_poster,
        "Nota_IMDb": nota,
        "Generos": lista_generos,
        "Diretores": lista_diretores,
        "URL": url
    })

    time.sleep(1)

driver.quit()

with open(ARQUIVO_JSON, "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

print(json.dumps(dados[:3], indent=2, ensure_ascii=False))
print("Arquivo JSON salvo com sucesso:", ARQUIVO_JSON)


Coletando 1/250: https://www.imdb.com/pt/title/tt0111161/?ref_=chttp_i_1
Coletando 2/250: https://www.imdb.com/pt/title/tt0068646/?ref_=chttp_i_2
Coletando 3/250: https://www.imdb.com/pt/title/tt0468569/?ref_=chttp_i_3
Coletando 4/250: https://www.imdb.com/pt/title/tt0071562/?ref_=chttp_i_4
Coletando 5/250: https://www.imdb.com/pt/title/tt0050083/?ref_=chttp_i_5
Coletando 6/250: https://www.imdb.com/pt/title/tt0167260/?ref_=chttp_i_6
Coletando 7/250: https://www.imdb.com/pt/title/tt0108052/?ref_=chttp_i_7
Coletando 8/250: https://www.imdb.com/pt/title/tt0120737/?ref_=chttp_i_8
Coletando 9/250: https://www.imdb.com/pt/title/tt0110912/?ref_=chttp_i_9
Coletando 10/250: https://www.imdb.com/pt/title/tt0060196/?ref_=chttp_i_10
Coletando 11/250: https://www.imdb.com/pt/title/tt0167261/?ref_=chttp_i_11
Coletando 12/250: https://www.imdb.com/pt/title/tt0109830/?ref_=chttp_i_12
Coletando 13/250: https://www.imdb.com/pt/title/tt0137523/?ref_=chttp_i_13
Coletando 14/250: https://www.imdb.com/pt/t